# Woche 10 · Agent-Loop, ReAct und Planung
**AI Engineering: von LLMs zu Agenten** · linkit e.V. am KIT

> **Ein Agent ist ein LLM, das in einer Schleife Tools nutzt, bis die Aufgabe erledigt ist.**

Bisher hat unser Assistent Fragen in ein oder zwei Schritten beantwortet. Heute bekommt er eine echte Aufgabe:

> *„Maschine M-07 meldet Überhitzung. Prüfe Status, Historie und Handbuch und schlage eine Maßnahme vor.“*

Welche Schritte nötig sind, hängt von den Zwischenergebnissen ab. Ihr baut den Agent-Loop selbst, mit Trace, Iterationslimit und Schleifenerkennung, lasst den Agenten nach dem **ReAct**-Muster denken und ergänzt einen **Planungsschritt**. Am Ende zeigt das Eval aus Woche 9, ob Planung hilft und was sie kostet.

**Aufbau:** A Workflow oder Agent? · B Der Loop · C ReAct · D Plan-then-Execute · E Messen
Pflicht: TODO 1–6, Ziel: TODO 7–8. API-Key nötig, Kosten: keine, die KI-Toolbox ist kostenlos (Fair Use).

In [ ]:
%pip install -q openai==3.24.0 rank-bm25==0.2.2

In [ ]:
import json, re, math, datetime, time
from typing import Literal
import pandas as pd
from openai import OpenAI
from pydantic import BaseModel, Field
from rank_bm25 import BM25Okapi

def check(name, test):
    """Selbsttest: zeigt ✅ oder ❌. Fehler im Test gelten als 'noch nicht gelöst'."""
    try:
        ok = bool(test())
    except Exception as e:
        print(f"❌ {name}: noch nicht gelöst ({type(e).__name__}: {e})")
        return
    print(f"✅ {name}" if ok else f"❌ {name}: Ergebnis stimmt noch nicht")

try:
    from google.colab import userdata
    API_KEY = userdata.get("KIT_TOOLBOX_KEY")
except ImportError:
    import os
    API_KEY = os.environ.get("KIT_TOOLBOX_KEY")
client = OpenAI(api_key=(API_KEY or "fehlt").strip(), base_url="https://ki-toolbox.scc.kit.edu/api/v1", max_retries=5)
MODEL, JUDGE_MODEL = "kit.mistral-small-4-119b-a8b", "kit.deepseek-v4-flash"
HEUTE = datetime.date.today()
print("Schlüssel geladen:", bool(API_KEY), "| heute:", HEUTE)

## Daten und Tools (vorgegeben, einfach ausführen)

Wie in Woche 9, plus ein **neues lesendes Tool** `get_sensor_data` mit aktuellen Messwerten und Grenzwerten. `create_ticket` wird heute nie ausgeführt, nur vorgeschlagen.

In [ ]:
MASCHINEN = [
    {'id': 'M-01', 'name': 'CNC-Fräszentrum 1', 'typ': 'Kessler KFZ 800', 'baujahr': 2018, 'standort': 'Halle 1, Platz 4', 'status': 'in Betrieb', 'betriebsstunden': 28450, 'letzte_wartung': '2026-08-12', 'service': 'Kessler Service, SV-1180'},
    {'id': 'M-02', 'name': 'CNC-Fräszentrum 2', 'typ': 'Kessler KFZ 800 plus', 'baujahr': 2021, 'standort': 'Halle 1, Platz 5', 'status': 'in Betrieb', 'betriebsstunden': 14120, 'letzte_wartung': '2026-09-02', 'service': 'Kessler Service, SV-1180'},
    {'id': 'M-03', 'name': 'CNC-Drehmaschine', 'typ': 'Hartwig TD 400', 'baujahr': 2015, 'standort': 'Halle 1, Platz 9', 'status': 'eingeschränkt', 'betriebsstunden': 41980, 'letzte_wartung': '2026-07-28', 'service': 'kein Servicevertrag'},
    {'id': 'M-04', 'name': 'Rundschleifmaschine', 'typ': 'Brenner RS 600', 'baujahr': 2019, 'standort': 'Halle 2, Platz 2', 'status': 'in Betrieb', 'betriebsstunden': 22310, 'letzte_wartung': '2026-09-15', 'service': 'kein Servicevertrag'},
    {'id': 'M-05', 'name': 'Roboter-Schweißzelle', 'typ': 'Zelle mit Varion 500', 'baujahr': 2020, 'standort': 'Halle 2, Platz 6', 'status': 'in Betrieb', 'betriebsstunden': 19870, 'letzte_wartung': '2026-09-09', 'service': 'SV-2045'},
    {'id': 'M-06', 'name': 'Montagelinie A', 'typ': 'Eigenbau, Umbau 2023', 'baujahr': 2017, 'standort': 'Halle 2, Bereich M', 'status': 'in Betrieb', 'betriebsstunden': 30560, 'letzte_wartung': '2026-08-30', 'service': 'kein Servicevertrag'},
    {'id': 'M-07', 'name': 'Prüfstand', 'typ': 'Druckprüfeinheit mit Messrechner', 'baujahr': 2022, 'standort': 'Halle 2, Bereich P', 'status': 'in Betrieb', 'betriebsstunden': 8740, 'letzte_wartung': '2026-06-20', 'service': 'Kalibrierung über QS'},
    {'id': 'M-08', 'name': 'Teilewaschanlage', 'typ': 'Durchlaufwaschanlage', 'baujahr': 2016, 'standort': 'Halle 1, Platz 12', 'status': 'in Betrieb', 'betriebsstunden': 33900, 'letzte_wartung': '2026-09-20', 'service': 'kein Servicevertrag'},
]
WARTUNGSHISTORIE = [
    ('M-01', '2026-08-12', 'Hydrauliköl gewechselt, Filter getauscht', 'Instandhaltung'),
    ('M-01', '2026-06-03', 'Abstreifer Führungsbahn X erneuert (KFZ-8810)', 'Instandhaltung'),
    ('M-01', '2026-03-18', 'Spindelrundlauf gemessen: 3 µm, in Ordnung', 'Instandhaltung'),
    ('M-02', '2026-09-02', 'Greifer des Werkzeugwechslers gereinigt, Federn getauscht (KFZ-9120)', 'Instandhaltung'),
    ('M-02', '2026-07-01', 'Messtaster kalibriert', 'Instandhaltung'),
    ('M-02', '2026-04-14', 'Kugelgewindetriebe geprüft, kein Spiel', 'Instandhaltung'),
    ('M-03', '2026-07-28', 'Spindellager nachgeschmiert (NW-LX2, 8 g je Lager)', 'Instandhaltung'),
    ('M-03', '2026-07-28', 'Spindelrundlauf gemessen: 7 µm, nahe am Grenzwert 8 µm', 'Instandhaltung'),
    ('M-03', '2026-06-30', 'Spannfutter zerlegt, gereinigt, Backen gefettet', 'Instandhaltung'),
    ('M-03', '2026-05-19', 'Spindellager nachgeschmiert (NW-LX2, 8 g je Lager)', 'Instandhaltung'),
    ('M-03', '2026-02-10', 'Hydrauliköl gewechselt, Rücklauffilter getauscht (HTD-3310)', 'Instandhaltung'),
    ('M-04', '2026-09-15', 'Neue Schleifscheibe montiert, Klangprobe in Ordnung, ausgewuchtet', 'Instandhaltung'),
    ('M-04', '2026-08-11', 'Wuchteinrichtung geprüft', 'Instandhaltung'),
    ('M-04', '2026-06-23', 'Schleifspindel auf Lagergeräusche geprüft, unauffällig', 'Instandhaltung'),
    ('M-05', '2026-09-09', 'Hydraulik der Spannvorrichtungen geprüft, kleine Leckage an Zylinder 2 abgedichtet', 'Instandhaltung'),
    ('M-05', '2026-08-05', 'Schweißstromquelle getauscht nach Ausfall', 'Servicepartner SV-2045'),
    ('M-05', '2026-05-12', 'Kühlflüssigkeit des Brenners ersetzt', 'Instandhaltung'),
    ('M-06', '2026-08-30', 'Kettenspannung Fördersystem nachgestellt', 'Instandhaltung'),
    ('M-06', '2026-07-14', 'Datensicherung der SPS nach Programmänderung', 'Instandhaltung'),
    ('M-06', '2026-06-02', 'Zentralschmierung aufgefüllt (NW-K100)', 'Instandhaltung'),
    ('M-07', '2026-06-20', 'Kalibrierung der Drucksensoren durch Dienstleister', 'Qualitätssicherung'),
    ('M-07', '2025-10-02', 'Elektrische Prüfung nach DGUV Vorschrift 3', 'Elektrofachkraft'),
    ('M-08', '2026-09-20', 'Waschbad gewechselt, Tank gereinigt', 'Instandhaltung'),
    ('M-08', '2026-08-24', 'Waschbad gewechselt, Tank gereinigt', 'Instandhaltung'),
    ('M-08', '2026-04-07', 'Heizstab entkalkt', 'Instandhaltung'),
]
TICKETS = [
    {'ticket': 'S-001', 'datum': '2026-10-05', 'schicht': 'Früh', 'maschine': 'M-03', 'kategorie': 'mechanisch', 'prioritaet': 'mittel', 'stillstand_min': 45, 'status': 'geschlossen', 'beschreibung': 'Spindel läuft unrund, Lager nachgestellt'},
    {'ticket': 'S-002', 'datum': '2026-10-05', 'schicht': 'Spät', 'maschine': 'M-01', 'kategorie': 'elektrisch', 'prioritaet': 'hoch', 'stillstand_min': 20, 'status': 'geschlossen', 'beschreibung': 'Not-Aus durch defekten Türschalter ausgelöst'},
    {'ticket': 'S-003', 'datum': '2026-10-05', 'schicht': 'Nacht', 'maschine': 'M-06', 'kategorie': 'Material', 'prioritaet': 'mittel', 'stillstand_min': 35, 'status': 'geschlossen', 'beschreibung': 'Fehlende Gehäuseteile an Station 3'},
    {'ticket': 'S-004', 'datum': '2026-10-06', 'schicht': 'Früh', 'maschine': 'M-05', 'kategorie': 'elektrisch', 'prioritaet': 'hoch', 'stillstand_min': 240, 'status': 'geschlossen', 'beschreibung': 'Schweißstromquelle ausgefallen, Tausch durch Instandhaltung'},
    {'ticket': 'S-005', 'datum': '2026-10-06', 'schicht': 'Spät', 'maschine': 'M-03', 'kategorie': 'Software', 'prioritaet': 'mittel', 'stillstand_min': 60, 'status': 'geschlossen', 'beschreibung': 'CNC-Programm nach Update nicht ladbar'},
    {'ticket': 'S-006', 'datum': '2026-10-06', 'schicht': 'Spät', 'maschine': 'M-02', 'kategorie': 'mechanisch', 'prioritaet': 'niedrig', 'stillstand_min': 25, 'status': 'geschlossen', 'beschreibung': 'Werkzeugwechsler klemmt'},
    {'ticket': 'S-007', 'datum': '2026-10-07', 'schicht': 'Früh', 'maschine': 'M-04', 'kategorie': 'Bedienung', 'prioritaet': 'niedrig', 'stillstand_min': 15, 'status': 'geschlossen', 'beschreibung': 'Falsches Schleifprogramm gewählt'},
    {'ticket': 'S-008', 'datum': '2026-10-07', 'schicht': 'Nacht', 'maschine': 'M-03', 'kategorie': 'mechanisch', 'prioritaet': 'mittel', 'stillstand_min': 35, 'status': 'geschlossen', 'beschreibung': 'Spänestau im Förderer'},
    {'ticket': 'S-009', 'datum': '2026-10-07', 'schicht': 'Nacht', 'maschine': 'M-07', 'kategorie': 'Software', 'prioritaet': 'mittel', 'stillstand_min': 50, 'status': 'geschlossen', 'beschreibung': 'Prüfsoftware hängt beim Messlauf'},
    {'ticket': 'S-010', 'datum': '2026-10-08', 'schicht': 'Früh', 'maschine': 'M-06', 'kategorie': 'Material', 'prioritaet': 'mittel', 'stillstand_min': 40, 'status': 'geschlossen', 'beschreibung': 'Schraubenzuführung leer'},
    {'ticket': 'S-011', 'datum': '2026-10-08', 'schicht': 'Spät', 'maschine': 'M-01', 'kategorie': 'mechanisch', 'prioritaet': 'mittel', 'stillstand_min': 70, 'status': 'geschlossen', 'beschreibung': 'Kühlmittelpumpe undicht'},
    {'ticket': 'S-012', 'datum': '2026-10-08', 'schicht': 'Spät', 'maschine': 'M-08', 'kategorie': 'elektrisch', 'prioritaet': 'mittel', 'stillstand_min': 30, 'status': 'geschlossen', 'beschreibung': 'Heizregister der Waschanlage ohne Funktion'},
    {'ticket': 'S-013', 'datum': '2026-10-09', 'schicht': 'Früh', 'maschine': 'M-03', 'kategorie': 'mechanisch', 'prioritaet': 'mittel', 'stillstand_min': 90, 'status': 'geschlossen', 'beschreibung': 'Spindel erneut unrund, Lagerwechsel geplant'},
    {'ticket': 'S-014', 'datum': '2026-10-09', 'schicht': 'Nacht', 'maschine': 'M-05', 'kategorie': 'Bedienung', 'prioritaet': 'mittel', 'stillstand_min': 30, 'status': 'geschlossen', 'beschreibung': 'Schweißdraht falsch eingefädelt'},
    {'ticket': 'S-015', 'datum': '2026-10-12', 'schicht': 'Früh', 'maschine': 'M-02', 'kategorie': 'elektrisch', 'prioritaet': 'mittel', 'stillstand_min': 55, 'status': 'geschlossen', 'beschreibung': 'Achsantrieb Y meldet Überstrom'},
    {'ticket': 'S-016', 'datum': '2026-10-12', 'schicht': 'Spät', 'maschine': 'M-06', 'kategorie': 'Bedienung', 'prioritaet': 'hoch', 'stillstand_min': 10, 'status': 'geschlossen', 'beschreibung': 'Lichtvorhang beim Teilewechsel unterbrochen'},
    {'ticket': 'S-017', 'datum': '2026-10-13', 'schicht': 'Früh', 'maschine': 'M-03', 'kategorie': 'elektrisch', 'prioritaet': 'mittel', 'stillstand_min': 40, 'status': 'geschlossen', 'beschreibung': 'Sensor am Spannfutter liefert kein Signal'},
    {'ticket': 'S-018', 'datum': '2026-10-13', 'schicht': 'Nacht', 'maschine': 'M-04', 'kategorie': 'mechanisch', 'prioritaet': 'hoch', 'stillstand_min': 65, 'status': 'geschlossen', 'beschreibung': 'Schleifscheibe gebrochen, Austausch'},
    {'ticket': 'S-019', 'datum': '2026-10-14', 'schicht': 'Früh', 'maschine': 'M-07', 'kategorie': 'Software', 'prioritaet': 'niedrig', 'stillstand_min': 20, 'status': 'geschlossen', 'beschreibung': 'Messprotokoll wird nicht gespeichert'},
    {'ticket': 'S-020', 'datum': '2026-10-14', 'schicht': 'Spät', 'maschine': 'M-01', 'kategorie': 'Material', 'prioritaet': 'niedrig', 'stillstand_min': 25, 'status': 'geschlossen', 'beschreibung': 'Rohteile mit Übermaß angeliefert'},
    {'ticket': 'S-021', 'datum': '2026-10-15', 'schicht': 'Früh', 'maschine': 'M-03', 'kategorie': 'mechanisch', 'prioritaet': 'mittel', 'stillstand_min': 55, 'status': 'offen', 'beschreibung': 'Spindelgeräusch, Lagerwechsel fürs Wochenende eingeplant'},
    {'ticket': 'S-022', 'datum': '2026-10-15', 'schicht': 'Spät', 'maschine': 'M-08', 'kategorie': 'mechanisch', 'prioritaet': 'hoch', 'stillstand_min': 45, 'status': 'geschlossen', 'beschreibung': 'Förderband der Waschanlage gerissen'},
    {'ticket': 'S-023', 'datum': '2026-10-16', 'schicht': 'Nacht', 'maschine': 'M-02', 'kategorie': 'Software', 'prioritaet': 'mittel', 'stillstand_min': 35, 'status': 'geschlossen', 'beschreibung': 'Werkzeugdaten nach Neustart verloren'},
    {'ticket': 'S-024', 'datum': '2026-10-16', 'schicht': 'Spät', 'maschine': 'M-06', 'kategorie': 'elektrisch', 'prioritaet': 'mittel', 'stillstand_min': 75, 'status': 'offen', 'beschreibung': 'Schrauber 2 liefert falsches Drehmoment'},
]
SENSORDATEN = {   # (Messgröße, Wert, Einheit, Grenzwert, Art des Grenzwerts)
    'M-01': [('Spindeltemperatur', 41, '°C', 70, 'max'), ('Vibration Spindel', 1.9, 'mm/s', 4.5, 'max'), ('Hydraulikdruck', 62, 'bar', 55, 'min')],
    'M-02': [('Spindeltemperatur', 44, '°C', 70, 'max'), ('Vibration Spindel', 2.3, 'mm/s', 4.5, 'max'), ('Hydraulikdruck', 60, 'bar', 55, 'min')],
    'M-03': [('Spindeltemperatur', 63, '°C', 70, 'max'), ('Vibration Spindel', 4.8, 'mm/s', 4.5, 'max'), ('Hydraulikdruck', 41, 'bar', 38, 'min')],
    'M-04': [('Spindeltemperatur', 38, '°C', 65, 'max'), ('Vibration Schleifspindel', 1.1, 'mm/s', 2.8, 'max'), ('Kühlmitteltemperatur', 22, '°C', 30, 'max')],
    'M-05': [('Kühlmitteltemperatur Brenner', 68, '°C', 60, 'max'), ('Schutzgas Flaschendruck', 74, 'bar', 10, 'min'), ('Hydraulikdruck Spannvorrichtung', 95, 'bar', 90, 'min')],
    'M-06': [('Kettenspannung', 0.92, 'kN', 0.8, 'min'), ('Taktzeit', 48, 's', 55, 'max'), ('Druckluft', 6.1, 'bar', 5.5, 'min')],
    'M-07': [('Temperatur Schaltschrank', 58, '°C', 45, 'max'), ('Drehzahl Schaltschranklüfter', 0, 'U/min', 1500, 'min'), ('Druckluft', 6.0, 'bar', 5.5, 'min')],
    'M-08': [('Temperatur Waschlauge', 52, '°C', 58, 'min'), ('Füllstand Waschtank', 84, '%', 70, 'min'), ('Förderbandgeschwindigkeit', 1.2, 'm/min', 1.0, 'min')],
}
HANDBUCH = '''# Wartungshandbuch Nordwerk GmbH, Werk Karlsruhe (Auszug)

Dokument WH-2026-03 · Stand: September 2026 · Herausgeber: Abteilung Instandhaltung · Dieses Dokument ist fiktiv und dient ausschließlich Schulungszwecken.

## 1 Allgemeines

### 1.1 Geltungsbereich und Aufbau

Dieses Handbuch gilt für alle Fertigungs- und Prüfanlagen in den Hallen 1 und 2 des Werks Karlsruhe. Es richtet sich an Maschinenbedienerinnen und -bediener, Schichtleitungen und das Instandhaltungsteam. Kapitel 1 regelt Zuständigkeiten, Sicherheit und Dokumentation. Kapitel 2 beschreibt Betriebsstoffe, die an mehreren Maschinen verwendet werden. Kapitel 3 enthält für jede Maschine eine Kurzbeschreibung, die Wartungsintervalle, typische Störungen und wichtige Ersatzteile. Kapitel 4 regelt das Ersatzteillager, Kapitel 5 enthält Fehlercodes und Kontakte.

Die Herstellerdokumentation der einzelnen Maschinen bleibt verbindlich. Bei Widersprüchen gilt die Herstellerangabe, und die Instandhaltung ist zu informieren, damit dieses Handbuch angepasst werden kann. Änderungen am Handbuch gibt die Leitung Instandhaltung frei. Die jeweils gültige Fassung liegt im Intranet. Ausgedruckte Exemplare an den Maschinen sind nach jeder Änderung auszutauschen.

### 1.2 Zuständigkeiten und Eskalation

Für die tägliche Pflege und die Sichtprüfungen ist die Bedienerin oder der Bediener der Maschine zuständig. Wartungsarbeiten mit Werkzeug, das Öffnen von Schaltschränken und alle Arbeiten an Hydraulik, Druckluftverteilung und Elektrik führt ausschließlich das Instandhaltungsteam durch.

Bei einer Störung meldet die Bedienerin oder der Bediener zunächst an die Schichtleitung. Die Schichtleitung entscheidet über die Priorität und informiert die Instandhaltung. Während der Kernzeit von 6:00 bis 22:00 Uhr ist die Instandhaltung über die Leitstelle erreichbar. Außerhalb der Kernzeit, an Wochenenden und an Feiertagen ist bei Störungen mit hoher Priorität die Rufbereitschaft der Instandhaltung unter der internen Nummer 4711 anzurufen. Bei Störungen mit mittlerer oder niedriger Priorität genügt außerhalb der Kernzeit ein Ticket, das am nächsten Arbeitstag bearbeitet wird.

Kann die Instandhaltung eine Störung nicht innerhalb von vier Stunden beheben, informiert sie die Produktionsleitung und entscheidet gemeinsam mit ihr, ob ein externer Servicepartner beauftragt wird. Die Kontakte der Servicepartner stehen in Kapitel 5.2.

### 1.3 Arbeitssicherheit

Vor jeder Wartung, Reinigung oder Störungsbehebung im Gefahrenbereich einer Maschine ist die Maschine freizuschalten. Das Vorgehen folgt immer denselben fünf Schritten: Maschine stillsetzen, Hauptschalter ausschalten, Hauptschalter mit persönlichem Vorhängeschloss gegen Wiedereinschalten sichern, Spannungsfreiheit und Druckfreiheit prüfen, Arbeitsbereich kennzeichnen. Jede Person, die im Gefahrenbereich arbeitet, hängt ihr eigenes Schloss an. Die Maschine darf erst wieder eingeschaltet werden, wenn alle Schlösser entfernt sind.

Arbeiten an Hydraulikanlagen sind nur im drucklosen Zustand erlaubt. Druckspeicher müssen vorher entladen werden, der Manometerwert muss null zeigen. Arbeiten in der Roboter-Schweißzelle erfolgen grundsätzlich zu zweit, eine Person bleibt außerhalb der Zelle und hat Zugriff auf den Not-Aus. Beim Umgang mit Kühlschmierstoff sind Schutzhandschuhe aus Nitril und eine Schutzbrille zu tragen. Hautschutzplan und Betriebsanweisung hängen an jeder Maschine mit Kühlschmierstoff aus.

Defekte Schutzeinrichtungen wie Türschalter, Lichtgitter oder Not-Aus-Taster dürfen niemals überbrückt werden. Ist eine Schutzeinrichtung defekt, bleibt die Maschine außer Betrieb, bis die Instandhaltung sie instand gesetzt und geprüft hat.

### 1.4 Dokumentation von Störungen und Wartungen

Jede Störung wird als Ticket im Instandhaltungssystem erfasst. Störungen mit hoher Priorität sind innerhalb von 30 Minuten nach Feststellung als Ticket anzulegen, alle anderen bis zum Ende der Schicht. Das Ticket enthält Maschine, Zeitpunkt, Beschreibung, vermutete Ursache, Priorität und Stillstandszeit.

Für die Priorität gelten im ganzen Werk dieselben Definitionen. Hoch bedeutet: Die Maschine steht oder es besteht ein Sicherheitsrisiko. Mittel bedeutet: Die Maschine läuft eingeschränkt, Qualität oder Tempo leiden. Niedrig bedeutet: kein Einfluss auf die laufende Produktion.

Durchgeführte Wartungen werden mit Datum, Tätigkeit, verbrauchten Ersatzteilen und Namen im Wartungsplan der Maschine eingetragen. Bei Abweichungen vom Sollzustand, etwa ungewöhnlichem Verschleiß, ist zusätzlich ein Hinweis-Ticket anzulegen, damit die Intervalle überprüft werden können.

## 2 Betriebsstoffe

### 2.1 Schmierstoffe und Öle

Im Werk werden nur die folgenden freigegebenen Betriebsstoffe verwendet. Andere Produkte dürfen nur nach Freigabe durch die Instandhaltung eingesetzt werden, weil falsche Schmierstoffe Dichtungen angreifen und Lager schädigen können.

| Anwendung | Bezeichnung | Gebinde | Lagerort |
|---|---|---|---|
| Hydraulikaggregate aller Maschinen | Hydrauliköl HLP 46 | 20-Liter-Kanister | Gefahrstofflager, Regal G-2 |
| Spindellager der Drehmaschine | Wälzlagerfett NW-LX2 | 400-Gramm-Kartusche | Ersatzteillager, Regal C-04 |
| Führungsbahnen der Fräszentren | Gleitbahnöl CGLP 68 | 20-Liter-Kanister | Gefahrstofflager, Regal G-3 |
| Zentralschmierung der Montagelinie | Kettenöl NW-K100 | 5-Liter-Kanister | Gefahrstofflager, Regal G-3 |
| Kühlschmierstoff aller Zerspanungsmaschinen | Emulsion NW-Cut 300 (Konzentrat) | 200-Liter-Fass | Gefahrstofflager, Bereich G-1 |

Leere Gebinde und Altöl werden in den gekennzeichneten Sammelbehältern im Gefahrstofflager entsorgt. Ölgetränkte Putzlappen gehören in die roten Sicherheitsbehälter an den Maschinen.

### 2.2 Pflege des Kühlschmierstoffs

Die Fräszentren, die Drehmaschine und die Rundschleifmaschine arbeiten mit der wassergemischten Emulsion NW-Cut 300. Ein gut gepflegter Kühlschmierstoff verlängert die Standzeit der Werkzeuge, verbessert die Oberflächengüte und schützt die Haut der Beschäftigten.

Die Konzentration soll zwischen 6 bis 8 Prozent liegen. Sie wird einmal pro Woche mit dem Refraktometer gemessen, der Messwert wird mit dem Faktor 1,1 multipliziert (Refraktometerfaktor des Konzentrats). Der pH-Wert soll zwischen 8,5 und 9,3 liegen, der Nitritgehalt unter 20 mg/l. Alle drei Werte werden im Kühlschmierstoff-Protokoll an der Maschine eingetragen.

Ist die Konzentration zu niedrig, wird Konzentrat über das Mischgerät nachdosiert, nie unverdünnt in den Tank gegeben. Ist sie zu hoch, wird mit Wasser nachgefüllt. Schäumt der Kühlschmierstoff stark, ist zuerst der Füllstand zu prüfen, denn zu wenig Flüssigkeit im Tank führt zu Luftansaugung an der Pumpe. Hilft das nicht, wird nach Rücksprache mit der Instandhaltung Entschäumer NW-AF 2 in einer Dosierung von 0,05 Prozent zugegeben. Riecht der Kühlschmierstoff faulig oder liegt der pH-Wert unter 8,0, ist er biologisch belastet und muss gewechselt werden. Unabhängig davon wird der Kühlschmierstoff spätestens alle sechs Monate vollständig gewechselt, einschließlich Reinigung des Tanks.

### 2.3 Druckluft

Die zentrale Druckluftversorgung liefert einen Netzdruck von 6 bar. An jeder Maschine sitzt eine Wartungseinheit aus Filter, Druckregler und Wasserabscheider. Der Wasserabscheider wird täglich zu Schichtbeginn entleert, sofern er keinen automatischen Ablass hat. Das Filterelement wird halbjährlich gewechselt oder früher, wenn die Differenzdruckanzeige rot zeigt.

Fällt der Druck an einer Maschine unter 5,5 bar, schaltet die Maschinensteuerung in den meisten Fällen mit einer Warnung ab. Druckschwankungen haben häufig eine von drei Ursachen: ein verschmutztes Filterelement, eine Leckage an Schläuchen oder Kupplungen oder ein zu hoher gleichzeitiger Verbrauch in der Halle. Leckagen lassen sich mit Lecksuchspray finden. Undichte Kupplungen werden getauscht, nicht abgedichtet.

## 3 Maschinen

### 3.1 M-01 CNC-Fräszentrum 1

#### Kurzbeschreibung

Das CNC-Fräszentrum 1 ist ein vertikales Bearbeitungszentrum vom Typ Kessler KFZ 800, Baujahr 2018, Standort Halle 1, Platz 4. Es bearbeitet vor allem Gehäuseteile aus Aluminium. Die Arbeitsspindel erreicht 12.000 Umdrehungen pro Minute. Das Werkzeugmagazin hat 24 Plätze. Verfahrwege: X 800 mm, Y 500 mm, Z 500 mm.

#### Wartungsintervalle

| Tätigkeit | Intervall | Zuständig |
|---|---|---|
| Späne aus dem Arbeitsraum entfernen, Sichtfenster reinigen | täglich | Bedienung |
| Füllstand Gleitbahnöl prüfen | täglich | Bedienung |
| Kühlschmierstoff messen (Kapitel 2.2) | wöchentlich | Bedienung |
| Abstreifer der Führungsbahnen prüfen | monatlich | Instandhaltung |
| Spindelrundlauf mit Prüfdorn messen, Grenzwert 5 µm | alle 1.000 Betriebsstunden | Instandhaltung |
| Hydrauliköl wechseln, Filter tauschen | jährlich | Instandhaltung |

#### Typische Störungen

| Symptom | Mögliche Ursache | Maßnahme |
|---|---|---|
| Achsantrieb meldet Überstrom | Schwergängige Führung, Späne unter dem Abstreifer, Motor defekt | Maschine freischalten, Abstreifer reinigen; tritt der Fehler erneut auf, Instandhaltung rufen |
| Kühlmittel tropft aus Leitung | Schlauchschelle locker, Schlauch porös | Auffangwanne unterstellen, Ticket mit Priorität niedrig, Schlauch beim nächsten Stillstand tauschen |
| CNC-Programm lässt sich nicht laden | Programm fehlerhaft übertragen, Speicher voll | Programm erneut aus der Programmverwaltung laden, alte Programme archivieren |

#### Wichtige Ersatzteile

| Teil | Teilenummer | Lagerort |
|---|---|---|
| Abstreifer Führungsbahn X | KFZ-8810 | Regal A-03 |
| Kühlmittelschlauch 12 mm, Meterware | NW-SL12 | Regal A-07 |

### 3.2 M-02 CNC-Fräszentrum 2

#### Kurzbeschreibung

Das CNC-Fräszentrum 2 ist baugleich zum Fräszentrum 1, jedoch vom neueren Typ Kessler KFZ 800 plus, Baujahr 2021, Standort Halle 1, Platz 5. Unterschiede: Das Werkzeugmagazin hat 40 Plätze statt 24, und die Maschine besitzt einen Messtaster zur automatischen Werkstückvermessung. Es fertigt vorwiegend Teile aus Stahl.

#### Wartungsintervalle

Es gelten dieselben Intervalle wie bei M-01. Zusätzlich gilt:

| Tätigkeit | Intervall | Zuständig |
|---|---|---|
| Greifer des Werkzeugwechslers reinigen und auf Verschleiß prüfen | monatlich | Instandhaltung |
| Messtaster kalibrieren | vierteljährlich | Instandhaltung |
| Kugelgewindetriebe auf Spiel prüfen | alle 2.000 Betriebsstunden | Instandhaltung |

#### Typische Störungen

| Symptom | Mögliche Ursache | Maßnahme |
|---|---|---|
| Werkzeugwechsler hakt, Wechsel dauert deutlich länger | Verschmutzte Greiferfinger, verschlissene Federn | Greifer reinigen; bei Verschleiß Federn tauschen (Teilenummer KFZ-9120) |
| Quietschen an einer Achse | Schmierung der Führung unzureichend | Zentralschmierung manuell auslösen, Gleitbahnöl-Füllstand prüfen; Geräusch weiter beobachten |
| Messtaster liefert stark streuende Werte | Tastkugel verschmutzt, Kalibrierung veraltet | Tastkugel reinigen, Messtaster neu kalibrieren |

### 3.3 M-03 CNC-Drehmaschine

#### Kurzbeschreibung

Die CNC-Drehmaschine ist eine Schrägbettmaschine vom Typ Hartwig TD 400, Baujahr 2015, Standort Halle 1, Platz 9. Sie fertigt Wellen und Bolzen bis 400 mm Länge. Das Hydraulikaggregat versorgt das Spannfutter und den Reitstock. Wegen ihres Alters ist sie die störanfälligste Maschine in Halle 1.

#### Wartungsintervalle

| Tätigkeit | Intervall | Zuständig |
|---|---|---|
| Späne entfernen, Spannfutter abblasen | täglich | Bedienung |
| Ölstand im Hydraulikaggregat prüfen (Hydrauliköl HLP 46) | täglich | Bedienung |
| Spindellager mit Wälzlagerfett NW-LX2 nachschmieren, Menge 8 Gramm je Lager | alle 500 Betriebsstunden | Instandhaltung |
| Spannfutter zerlegen, reinigen, Backen fetten | alle 250 Betriebsstunden | Instandhaltung |
| Spindelrundlauf messen, Grenzwert 8 µm | alle 1.000 Betriebsstunden | Instandhaltung |
| Hydrauliköl wechseln, Rücklauffilter tauschen | jährlich | Instandhaltung |

#### Typische Störungen

| Symptom | Mögliche Ursache | Maßnahme |
|---|---|---|
| Spindel vibriert, Teile außerhalb der Toleranz | Spindellager verschlissen oder unzureichend geschmiert | Drehzahl reduzieren, Instandhaltung informieren; Rundlauf messen, Lager nachschmieren oder tauschen |
| Maschine schaltet mit Meldung „Hydraulikdruck“ ab | Ölstand unter Minimum, Leckage | Ölstand prüfen; nur mit HLP 46 auffüllen; Leckage suchen, Ticket mit Priorität hoch |
| Spannfutter spannt ungleichmäßig | Späne in der Backenführung | Spannfutter reinigen; bei Wiederholung zerlegen |

#### Wichtige Ersatzteile

| Teil | Teilenummer | Lagerort |
|---|---|---|
| Ersatz-Spannfutter dreibackig, 250 mm | HTD-2250 | Regal C-12 |
| Spindellagersatz | HTD-1100 | Regal C-11 |
| Rücklauffilter Hydraulik | HTD-3310 | Regal C-04 |

### 3.4 M-04 Rundschleifmaschine

#### Kurzbeschreibung

Die Rundschleifmaschine vom Typ Brenner RS 600, Baujahr 2019, Standort Halle 2, Platz 2, schleift gehärtete Wellen auf das Endmaß. Die Schleifscheibe hat einen Durchmesser von 500 mm. Der Kühlschmierstoff wird über ein Papierbandfilter gereinigt.

#### Sicherheitshinweise zur Schleifscheibe

Schleifscheiben sind bruchgefährdet. Eine neue Schleifscheibe wird vor der Montage durch eine Klangprobe geprüft: Die Scheibe wird frei hängend mit einem Holzhammer leicht angeschlagen. Ein heller, klarer Klang zeigt eine unbeschädigte Scheibe, ein dumpfer oder scheppernder Klang deutet auf Risse hin. Scheiben mit Rissen oder unklarer Klangprobe dürfen nicht montiert werden. Die zulässige Umfangsgeschwindigkeit der eingesetzten Scheiben beträgt höchstens 50 m/s. Nach jeder Montage wird die Scheibe ausgewuchtet und abgerichtet, anschließend läuft sie mindestens fünf Minuten mit Betriebsdrehzahl, während sich niemand in der Drehebene aufhält.

#### Wartungsintervalle

| Tätigkeit | Intervall | Zuständig |
|---|---|---|
| Schleifscheibe abrichten | je nach Bedarf, spätestens alle 50 Werkstücke | Bedienung |
| Papierbandfilter prüfen, Rolle wechseln | wenn die Füllstandsanzeige gelb zeigt | Bedienung |
| Kühlschmierstoff messen (Kapitel 2.2) | wöchentlich | Bedienung |
| Wuchteinrichtung prüfen | monatlich | Instandhaltung |
| Schleifspindel auf Lagergeräusche prüfen | vierteljährlich | Instandhaltung |

#### Typische Störungen

| Symptom | Mögliche Ursache | Maßnahme |
|---|---|---|
| Risse an der Schleifscheibe | Schlag, Überlastung, falsche Lagerung | Maschine sofort stoppen und freischalten, Scheibe tauschen, Ticket mit Priorität hoch |
| Rattermarken auf dem Werkstück | Scheibe unwuchtig oder stumpf | Scheibe auswuchten und abrichten |
| Englische Bedienoberfläche nach Update | Spracheinstellung zurückgesetzt | Im Menü Einstellungen, Sprache auf Deutsch stellen; kein Einfluss auf die Produktion |

### 3.5 M-05 Roboter-Schweißzelle

#### Kurzbeschreibung

Die Roboter-Schweißzelle besteht aus einem Sechsachs-Roboter, der Schweißstromquelle Varion 500, einem Drehtisch mit zwei hydraulischen Spannvorrichtungen und einem Lichtgitter an der Beladeseite. Baujahr 2020, Standort Halle 2, Platz 6. Geschweißt wird im MAG-Verfahren mit Mischgas.

#### Wartungsintervalle

| Tätigkeit | Intervall | Zuständig |
|---|---|---|
| Brenner reinigen, Gasdüse und Kontaktrohr prüfen | alle vier Stunden Schweißbetrieb | Bedienung |
| Lichtgitter mit Prüfstab testen | täglich zu Schichtbeginn | Bedienung |
| Schutzgasflasche wechseln | wenn der Flaschendruck unter 10 bar fällt | Bedienung |
| Kühlflüssigkeit des Brenners prüfen | wöchentlich | Bedienung |
| Hydraulik der Spannvorrichtungen auf Leckagen prüfen | monatlich | Instandhaltung |
| Roboter-Getriebeöl wechseln | alle 20.000 Betriebsstunden | Servicepartner |

#### Typische Störungen

| Symptom | Mögliche Ursache | Maßnahme |
|---|---|---|
| Schweißstromquelle zeigt Fehlercode | siehe Kapitel 5.1 | Fehlercode ablesen und nach Tabelle vorgehen |
| Lichtgitter löst aus, obwohl sich niemand im Schutzfeld befindet | Verschmutzte Linsen, Schweißspritzer auf Sender oder Empfänger, verstellte Ausrichtung | Linsen mit weichem Tuch reinigen, Ausrichtung prüfen; Lichtgitter niemals überbrücken; bei Wiederholung Instandhaltung rufen |
| Spannvorrichtung öffnet sich während des Schweißens | Hydraulikdruck fällt ab, Leckage am Spannzylinder | Zelle sofort stillsetzen, Ticket mit Priorität hoch, Verletzungsgefahr |
| Bildschirm am Bedienpult flackert | Wackelkontakt im Displaykabel | Ticket mit Priorität niedrig, Bedienung bleibt möglich |

### 3.6 M-06 Montagelinie A

#### Kurzbeschreibung

Die Montagelinie A verbindet sechs Montagestationen über ein Werkstückträger-Fördersystem. Gesteuert wird sie von einer zentralen SPS, die über das Werksnetz mit dem Leitrechner der Produktion verbunden ist. Lichtschranken an der Zuführung erkennen die Werkstückträger. Baujahr 2017, Umbau 2023, Standort Halle 2, Bereich M.

#### Wartungsintervalle

| Tätigkeit | Intervall | Zuständig |
|---|---|---|
| Lichtschranken an Zuführung und Stationen reinigen | täglich | Bedienung |
| Not-Aus-Taster aller Stationen auf Funktion prüfen | wöchentlich | Schichtleitung |
| Kettenspannung des Fördersystems prüfen | monatlich | Instandhaltung |
| Zentralschmierung auffüllen (Kettenöl NW-K100) | wenn die Anzeige rot zeigt | Instandhaltung |
| Datensicherung der SPS | nach jeder Programmänderung, mindestens vierteljährlich | Instandhaltung |

#### Typische Störungen

| Symptom | Mögliche Ursache | Maßnahme |
|---|---|---|
| Not-Aus lässt sich nicht zurücksetzen | Taster mechanisch verklemmt, Fehler im Sicherheitskreis | Alle Not-Aus-Taster der Linie prüfen; bleibt der Fehler, Instandhaltung rufen; Linie steht, Priorität hoch |
| SPS verliert die Verbindung zum Leitrechner | Netzwerkkabel, Switch, IP-Konflikt | Linkanzeige am Switch prüfen; Neustart der SPS nur durch die Instandhaltung |
| Kurze Unterbrechungen durch Lichtschranke | Staub auf der Optik | Lichtschranke reinigen, bei Wiederholung Ausrichtung prüfen |

### 3.7 M-07 Prüfstand

#### Kurzbeschreibung

Der Prüfstand prüft fertig montierte Baugruppen auf Dichtheit und Funktion. Er besteht aus einer Druckprüfeinheit mit Druckluft, einem Messrechner mit Prüfsoftware und einem Etikettendrucker. Baujahr 2022, Standort Halle 2, Bereich P. Die Messergebnisse werden automatisch auf dem Qualitätsserver gespeichert.

#### Wartungsintervalle

| Tätigkeit | Intervall | Zuständig |
|---|---|---|
| Prüfadapter und Dichtungen auf Beschädigung prüfen | täglich | Bedienung |
| Leckprüfung mit Referenzteil | zu jedem Schichtbeginn | Bedienung |
| Kalibrierung der Drucksensoren | jährlich durch einen akkreditierten Dienstleister | Qualitätssicherung |
| Elektrische Prüfung nach DGUV Vorschrift 3 | jährlich | Elektrofachkraft |
| Datensicherung des Messrechners prüfen | monatlich | IT |

#### Typische Störungen

| Symptom | Mögliche Ursache | Maßnahme |
|---|---|---|
| Messprotokolle werden nicht gespeichert | Verbindung zum Qualitätsserver gestört, Speicher voll | Werte von Hand im Ersatzprotokoll notieren, IT informieren, Priorität mittel |
| Messungen streuen, Referenzteil fällt durch | Druckluft schwankt, Dichtung am Adapter verschlissen | Netzdruck an der Wartungseinheit prüfen (Kapitel 2.3), Dichtung tauschen |
| Prüfplakette abgelaufen | Prüftermin verpasst | Elektrofachkraft informieren, Prüfung nachholen |

### 3.8 M-08 Teilewaschanlage

#### Kurzbeschreibung

Die Teilewaschanlage reinigt Teile nach der Zerspanung von Spänen und Kühlschmierstoff. Sie arbeitet mit wässriger, alkalischer Waschlauge bei 60 °C, die über einen elektrischen Heizstab erwärmt wird. Ein Förderband transportiert die Teilekörbe durch Wasch-, Spül- und Trockenzone. Baujahr 2016, Standort Halle 1, Platz 12.

#### Wartungsintervalle

| Tätigkeit | Intervall | Zuständig |
|---|---|---|
| Siebe in der Waschzone reinigen | täglich | Bedienung |
| Temperatur und Füllstand der Waschlauge prüfen | täglich | Bedienung |
| Ölabscheider leeren | wöchentlich | Bedienung |
| Waschbad komplett wechseln, Tank reinigen | alle vier Wochen | Instandhaltung |
| Förderband auf Spannung und Verschleiß prüfen | monatlich | Instandhaltung |
| Heizstab entkalken | halbjährlich | Instandhaltung |

#### Typische Störungen

| Symptom | Mögliche Ursache | Maßnahme |
|---|---|---|
| Teile bleiben fettig | Waschlauge zu kalt oder verbraucht | Temperatur prüfen; liegt sie deutlich unter 60 °C, Heizstab und Sicherung prüfen lassen; Bad wechseln, wenn es älter als vier Wochen ist |
| Förderband steht | Band gerissen oder Antrieb blockiert | Anlage freischalten, Ursache prüfen, Priorität hoch |
| Waschzyklen zu kurz nach Stromausfall | Zeitsteuerung auf Werkseinstellung zurückgesetzt | Parameter aus der Parameterliste im Schaltschrank neu eingeben |

## 4 Ersatzteillager und Bestellung

Das Ersatzteillager befindet sich in Halle 2 neben der Werkstatt der Instandhaltung. Die Regale sind nach Maschinen gruppiert: Regalreihe A für die Fräszentren, B für die Schleifmaschine, C für die Drehmaschine, D für die Schweißzelle, E für Montagelinie und Prüfstand, F für die Waschanlage. Gefahrstoffe lagern getrennt im Gefahrstofflager.

Für kritische Teile gelten Mindestbestände. Wird ein Teil entnommen, trägt die entnehmende Person die Entnahme in die Lagerliste ein. Fällt der Bestand unter den Mindestbestand, bestellt die Instandhaltung über den Einkauf nach. Bestellungen bis 500 Euro gibt die Leitung Instandhaltung frei, darüber die Werkleitung. Für Eilbestellungen bei Stillstand mit hoher Priorität darf die Rufbereitschaft auch außerhalb der Kernzeit direkt beim Servicepartner bestellen und dokumentiert dies am nächsten Arbeitstag.

| Teil | Mindestbestand | Übliche Lieferzeit |
|---|---|---|
| Schleifscheibe 500 mm für M-04 | 2 Stück | 3 Wochen |
| Spindellagersatz HTD-1100 für M-03 | 1 Satz | 6 Wochen |
| Kontaktrohre für den Schweißbrenner | 50 Stück | 1 Woche |
| Lichtschranke Zuführung M-06 | 1 Stück | 2 Wochen |

## 5 Anhang

### 5.1 Fehlercodes der Schweißstromquelle Varion 500

| Code | Bedeutung | Maßnahme |
|---|---|---|
| E10 | Netzspannung außerhalb des zulässigen Bereichs | Netzversorgung prüfen lassen |
| E21 | Übertemperatur im Kühlkreislauf des Brenners | Kühlflüssigkeit prüfen und auffüllen, Kühler auf Verschmutzung prüfen; Anlage abkühlen lassen |
| E33 | Drahtvorschub blockiert | Drahtführung und Vorschubrollen prüfen, Drahtspule auf Verwicklungen prüfen |
| E41 | Kein Gasdurchfluss | Flaschenventil und Druckminderer prüfen, Flasche bei unter 10 bar wechseln |
| E45 | Kommunikation zur Robotersteuerung unterbrochen | Steckverbindung des Buskabels prüfen, Instandhaltung rufen |

### 5.2 Kontakte

| Bereich | Kontakt |
|---|---|
| Leitstelle Instandhaltung (Kernzeit 6 bis 22 Uhr) | intern 4700 |
| Rufbereitschaft Instandhaltung (außerhalb der Kernzeit) | intern 4711 |
| IT-Service | intern 4800 |
| Servicepartner Fräszentren (Kessler Service) | Servicevertrag SV-1180, Hotline im Intranet |
| Servicepartner Schweißzelle | Servicevertrag SV-2045, Hotline im Intranet |
| Kalibrierdienstleister Prüfstand | Auftrag über die Qualitätssicherung |
'''

GUELTIGE_IDS = [m["id"] for m in MASCHINEN]
def _maschine(machine_id):
    for m in MASCHINEN:
        if m["id"] == machine_id:
            return m
    raise ValueError(f"Unbekannte Maschinen-ID '{machine_id}'. Gültig sind: " + ", ".join(f"{m['id']} ({m['name']})" for m in MASCHINEN) + ".")

def get_machine_status(machine_id):
    m = _maschine(machine_id)
    offen = [t["ticket"] + ": " + t["beschreibung"] for t in TICKETS if t["maschine"] == machine_id and t["status"] == "offen"]
    return {"id": m["id"], "name": m["name"], "status": m["status"], "betriebsstunden": m["betriebsstunden"],
            "letzte_wartung": m["letzte_wartung"], "offene_tickets": offen}

def get_maintenance_history(machine_id, limit=5):
    _maschine(machine_id)
    eintraege = sorted((w for w in WARTUNGSHISTORIE if w[0] == machine_id), key=lambda w: w[1], reverse=True)
    return [{"datum": d, "taetigkeit": t, "durchgefuehrt_von": v} for _, d, t, v in eintraege[:limit]]

def get_sensor_data(machine_id):
    _maschine(machine_id)
    def bewertung(wert, grenze, art):
        return "ÜBERSCHRITTEN" if (art == "max" and wert > grenze) or (art == "min" and wert < grenze) else "ok"
    return [{"messgroesse": n, "wert": w, "einheit": e, "grenzwert": f"{'max.' if a == 'max' else 'min.'} {g}", "bewertung": bewertung(w, g, a)}
            for n, w, e, g, a in SENSORDATEN[machine_id]]

def chunk_struktur(markdown):
    chunks, pfad, zeilen, titel = [], {}, [], ""
    def abschliessen():
        inhalt = "\n".join(zeilen).strip()
        if inhalt:
            chunks.append({"titel": titel, "text": inhalt})
    for zeile in markdown.splitlines():
        if zeile.startswith("#"):
            abschliessen()
            ebene = len(zeile) - len(zeile.lstrip("#"))
            pfad = {e: t for e, t in pfad.items() if e < ebene}
            pfad[ebene] = zeile.lstrip("#").strip()
            titel = " › ".join(pfad[e] for e in sorted(pfad) if e > 1) or pfad[ebene]
            zeilen = []
        else:
            zeilen.append(zeile)
    abschliessen()
    return chunks

CHUNKS = chunk_struktur(HANDBUCH)
_tok = lambda t: re.findall(r"[a-zäöüß0-9-]+", t.lower())
_bm25 = BM25Okapi([_tok(c["titel"] + " " + c["text"]) for c in CHUNKS])
def search_knowledge_base(query, k=3):
    scores = _bm25.get_scores(_tok(query))
    besten = sorted(range(len(CHUNKS)), key=lambda i: scores[i], reverse=True)[:k]
    return [{"quelle": CHUNKS[i]["titel"], "text": CHUNKS[i]["text"]} for i in besten]

VORGESCHLAGENE_TICKETS = []
def create_ticket(machine_id, description, priority):
    _maschine(machine_id)
    VORGESCHLAGENE_TICKETS.append({"maschine": machine_id, "beschreibung": description, "prioritaet": priority})
    return {"status": "als Vorschlag notiert, ein Mensch muss es freigeben"}

ID = {"type": "string", "description": "Maschinen-ID im Format M-01 bis M-08."}
TOOLS = [
    {"name": "get_machine_status", "description": "Aktueller Zustand einer Maschine: Status, Betriebsstunden, letzte Wartung, offene Tickets mit Beschreibung.",
     "parameters": {"type": "object", "properties": {"machine_id": ID}, "required": ["machine_id"], "additionalProperties": False}},
    {"name": "get_maintenance_history", "description": "Letzte durchgeführte Wartungen einer Maschine, neueste zuerst, mit Datum und Tätigkeit.",
     "parameters": {"type": "object", "properties": {"machine_id": ID, "limit": {"type": "integer"}}, "required": ["machine_id"]}},
    {"name": "get_sensor_data", "description": "Aktuelle Messwerte einer Maschine (Mittel der letzten Stunde) mit Grenzwert und Bewertung ok oder ÜBERSCHRITTEN. Für Diagnose von Auffälligkeiten.",
     "parameters": {"type": "object", "properties": {"machine_id": ID}, "required": ["machine_id"], "additionalProperties": False}},
    {"name": "search_knowledge_base", "description": "Durchsucht das Wartungshandbuch: Intervalle, Sicherheit, Störungsbehebung, Betriebsstoffe, Ersatzteile, Fehlercodes, Zuständigkeiten.",
     "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}},
    {"name": "create_ticket", "description": "Schlägt ein Störungsticket vor. Ein Mensch gibt es frei. Nur wenn eine Störung eindeutig festgestellt ist.",
     "parameters": {"type": "object", "properties": {"machine_id": ID, "description": {"type": "string"}, "priority": {"type": "string", "enum": ["hoch", "mittel", "niedrig"]}},
                      "required": ["machine_id", "description", "priority"], "additionalProperties": False}},
]
FUNKTIONEN = {"get_machine_status": get_machine_status, "get_maintenance_history": get_maintenance_history, "get_sensor_data": get_sensor_data,
              "search_knowledge_base": search_knowledge_base, "create_ticket": create_ticket}

def fuehre_tool_aus(name, eingabe):
    try:
        if name not in FUNKTIONEN:
            raise ValueError(f"Unbekanntes Tool '{name}'. Verfügbar sind: {', '.join(FUNKTIONEN)}.")
        return FUNKTIONEN[name](**eingabe), False
    except Exception as e:
        return {"fehler": str(e)}, True

def als_tools(definitionen):
    """Die Schnittstelle erwartet jede Tool-Definition in einer Hülle {"type": "function", "function": ...}."""
    return [{"type": "function", "function": d} for d in definitionen]

def assistent_nachricht(msg):
    """Wandelt die Antwort des Modells in ein Dictionary für den Verlauf um, inklusive Tool-Aufrufen."""
    d = {"role": "assistant", "content": msg.content or ""}
    if msg.tool_calls:
        d["tool_calls"] = [{"id": tc.id, "type": "function",
                            "function": {"name": tc.function.name, "arguments": tc.function.arguments}} for tc in msg.tool_calls]
    return d

def argumente(tc):
    """Die Argumente eines Tool-Aufrufs kommen als JSON-String."""
    try:
        return json.loads(tc.function.arguments or "{}")
    except json.JSONDecodeError:
        return {"_ungueltig": tc.function.arguments}

def tool_nachricht(tool_call_id, ergebnis, is_error=False):
    inhalt = json.dumps(ergebnis, ensure_ascii=False)
    return {"role": "tool", "tool_call_id": tool_call_id, "content": ("FEHLER: " + inhalt) if is_error else inhalt}

def kosten_cent(tokens):
    """Was die Tokens bei einem kommerziellen Anbieter zum Referenzpreis kosten würden (1 $ / 5 $ pro Mio.)."""
    return (tokens["input"] * 1.0 + tokens["output"] * 5.0) / 1e6 * 100
print("Bereit:", [t["name"] for t in TOOLS])

---
# Teil A · Workflow oder Agent?

Anthropic unterscheidet in „Building Effective AI Agents“ zwei Arten von Systemen:
- **Workflow:** Der **Code** legt die Schritte fest, das LLM füllt einzelne Schritte aus.
- **Agent:** Das **LLM** entscheidet selbst, welche Schritte es in welcher Reihenfolge braucht.

Workflows sind billiger, schneller und vorhersagbarer. Agenten sind flexibler, wenn sich die nötigen Schritte nicht vorab festlegen lassen. Faustregel: **So einfach wie möglich.** Ein Agent lohnt sich erst, wenn ein Workflow nicht reicht.

### TODO 1 · Ein Workflow
Schreibt `workflow_bericht(machine_id)`. Der **Code** ruft nacheinander `get_machine_status`, `get_maintenance_history` und `get_sensor_data` auf. Danach fasst **ein einziger** LLM-Aufruf die drei Ergebnisse in drei Sätzen zusammen. Gebt den Text zurück.

In [ ]:
def workflow_bericht(machine_id):
    # TODO 1
    pass

In [ ]:
bericht = workflow_bericht("M-07")
check("TODO 1 Workflow liefert einen Bericht", lambda: isinstance(bericht, str) and len(bericht) > 30)
if bericht:
    print(bericht)

> **Zum Nachdenken:** Der Workflow funktioniert für „Gib mir einen Zustandsbericht zu M-07“. Was wäre, wenn die Aufgabe lautet „Finde heraus, warum die Teile fettig sind“? Welche Schritte wären dann nötig, und ließen sie sich vorab festlegen?

---
# Teil B · Der Agent-Loop

Der Loop ist die Tool-Schleife aus Woche 8, ergänzt um drei Dinge, die ein Agent braucht:
1. **Trace:** Jeder Schritt wird protokolliert, damit man nachvollziehen kann, was passiert ist.
2. **Iterationslimit:** Nach `max_schritte` Runden ist Schluss.
3. **Schleifenerkennung:** Ruft der Agent mehrfach exakt dasselbe Tool mit denselben Parametern auf, bricht er ab.

### TODO 2 · Schleifen erkennen
`wiederholt_sich(bisherige_aufrufe, name, eingabe, max_gleich=2)` gibt `True` zurück, wenn derselbe Aufruf (gleicher Name, gleiche Eingabe) schon `max_gleich`-mal in `bisherige_aufrufe` vorkommt. `bisherige_aufrufe` ist eine Liste von Paaren `(name, eingabe)`.
Tipp: Eingaben vergleicht man am einfachsten als `json.dumps(eingabe, sort_keys=True)`.

In [ ]:
def wiederholt_sich(bisherige_aufrufe, name, eingabe, max_gleich=2):
    # TODO 2
    pass

In [ ]:
_a = [("get_sensor_data", {"machine_id": "M-07"}), ("get_sensor_data", {"machine_id": "M-07"}), ("get_sensor_data", {"machine_id": "M-03"})]
check("TODO 2a zweimal gleich → Schleife", lambda: wiederholt_sich(_a, "get_sensor_data", {"machine_id": "M-07"}) is True)
check("TODO 2b einmal → keine Schleife", lambda: wiederholt_sich(_a, "get_sensor_data", {"machine_id": "M-03"}) is False)
check("TODO 2c Reihenfolge der Schlüssel egal", lambda: wiederholt_sich([("x", {"a": 1, "b": 2})] * 2, "x", {"b": 2, "a": 1}) is True)

### TODO 3 · Der Loop
Ergänzt die markierten Stellen in `agent`. Der Trace ist eine Liste von Einträgen `{"schritt", "art", "inhalt", "tokens"}` mit den Arten:
- `Gedanke`: Text des Modells vor einem Tool-Aufruf
- `Aktion`: ein Tool-Aufruf
- `Beobachtung`: das Ergebnis
- `Antwort`: die finale Antwort
- `Abbruch`: Limit oder Schleife

Die Hilfsfunktion `notiere` ist vorgegeben. Gebt am Ende ein Dictionary mit `antwort`, `status` (`fertig`, `limit` oder `schleife`), `trace`, `tokens` und `schritte` zurück.

In [ ]:
def notiere(trace, schritt, art, inhalt, tokens=0):
    trace.append({"schritt": schritt, "art": art, "inhalt": inhalt if isinstance(inhalt, str) else json.dumps(inhalt, ensure_ascii=False), "tokens": tokens})

def agent(aufgabe, system, max_schritte=10, erste_nachricht=None):
    nachrichten = [{"role": "system", "content": system}, {"role": "user", "content": erste_nachricht or aufgabe}]
    trace, aufrufe, tokens = [], [], {"input": 0, "output": 0}
    for schritt in range(1, max_schritte + 1):
        antwort = client.chat.completions.create(model=MODEL, max_tokens=1000, tools=als_tools(TOOLS), messages=nachrichten)
        tokens["input"] += antwort.usage.prompt_tokens; tokens["output"] += antwort.usage.completion_tokens
        msg = antwort.choices[0].message
        nachrichten.append(assistent_nachricht(msg))
        text = (msg.content or "").strip()

        # TODO 3a: Enthält die Antwort keine tool_calls, Text als "Antwort" notieren und das Ergebnis mit status "fertig" zurückgeben.
        #          Sonst einen vorhandenen Text als "Gedanke" notieren.

        for tc in msg.tool_calls or []:
            name, eingabe = tc.function.name, argumente(tc)
            # TODO 3b: Bei Wiederholung "Abbruch" notieren und mit status "schleife" zurückgeben.
            #          Sonst "Aktion" notieren, Tool ausführen, Aufruf merken, "Beobachtung" notieren,
            #          tool_nachricht(tc.id, ergebnis, fehler) an nachrichten anhängen.
            pass

    notiere(trace, max_schritte, "Abbruch", f"Iterationslimit von {max_schritte} Schritten erreicht")
    return {"antwort": None, "status": "limit", "trace": trace, "tokens": tokens, "schritte": max_schritte}

### TODO 4 · Trace als Tabelle
`trace_tabelle(trace, breite=110)` macht aus dem Trace einen pandas-`DataFrame` mit den Spalten `Schritt`, `Art`, `Inhalt` und `Tokens`. Kürzt den Inhalt auf `breite` Zeichen (mit „…“ am Ende, wenn gekürzt wurde).

In [ ]:
def trace_tabelle(trace, breite=110):
    # TODO 4
    pass

In [ ]:
_t = []; notiere(_t, 1, "Aktion", "x" * 200); notiere(_t, 1, "Beobachtung", {"a": 1})
check("TODO 4a Spalten", lambda: list(trace_tabelle(_t).columns) == ["Schritt", "Art", "Inhalt", "Tokens"])
check("TODO 4b Kürzung", lambda: len(trace_tabelle(_t, 50).iloc[0]["Inhalt"]) == 50 and trace_tabelle(_t, 50).iloc[0]["Inhalt"].endswith("…"))

SYSTEM_BASIS = f"""Du bist der Instandhaltungs-Agent der Nordwerk GmbH. Heute ist der {HEUTE:%d.%m.%Y}.
Nutze die Tools, um die Aufgabe vollständig zu erledigen. Erfinde keine Fakten. Antworte am Ende knapp auf Deutsch."""

AUFGABE_M07 = "Maschine M-07 meldet Überhitzung. Prüfe Status, Historie und Handbuch und schlage eine Maßnahme vor."
try:
    erg = agent(AUFGABE_M07, SYSTEM_BASIS)
except Exception as e:
    erg = None
    print("⏳ Erst TODO 3 lösen:", type(e).__name__)
if erg and erg["trace"] and trace_tabelle([]) is not None:
    pd.set_option("display.max_colwidth", 120)
    display(trace_tabelle(erg["trace"]))
    print(f"\nStatus: {erg['status']} | Schritte: {erg['schritte']} | Tokens: {erg['tokens']} | Referenzkosten: {kosten_cent(erg['tokens']):.2f} Cent")
    print("\n💬", erg["antwort"])
check("TODO 3 Agent erledigt die Aufgabe", lambda: erg["status"] == "fertig" and any(t["art"] == "Aktion" for t in erg["trace"]))

> **Traces lesen:** In welcher Reihenfolge hat der Agent die Tools aufgerufen? Hat er etwas Überflüssiges getan? Hat er die überfällige elektrische Prüfung gefunden? Steht im Handbuch überhaupt etwas zu Überhitzung am Prüfstand?

**Typische Fehler eines Agenten, live geprüft:**

In [ ]:
# Iterationslimit: Was passiert bei einer Aufgabe, die viele Schritte braucht, aber wenig Spielraum hat?
try:
    knapp = agent("Prüfe die Sensordaten aller acht Maschinen und nenne alle Auffälligkeiten.", SYSTEM_BASIS, max_schritte=2)
except Exception:
    knapp = None
if knapp and trace_tabelle([]) is not None:
    print("Status:", knapp["status"], "| Schritte:", knapp["schritte"])
    display(trace_tabelle(knapp["trace"]).tail(4))

---
# Teil C · ReAct: Denken, Handeln, Beobachten

ReAct (Yao et al. 2022) lässt das Modell **vor jeder Aktion** kurz begründen, was es weiß, was fehlt und was es als Nächstes tut, und **nach jeder Beobachtung** prüfen, ob das Ergebnis den Plan ändert. Das macht Agenten oft zielgerichteter, und der Trace wird lesbar.

### TODO 5 · ReAct-System-Prompt
Schreibt `SYSTEM_REACT`. Er soll verlangen:
- vor jedem Tool-Aufruf einen kurzen **Gedanken** (ein bis zwei Sätze: Was weiß ich? Was fehlt? Welches Tool als Nächstes?)
- nach jeder Beobachtung zu prüfen, ob sie die Annahmen ändert
- aufzuhören, sobald die Aufgabe erledigt ist, und **ehrlich zu sagen, was offen bleibt**
- eine Antwort mit den Abschnitten **Befund**, **Ursache** und **Maßnahme**

In [ ]:
# TODO 5
SYSTEM_REACT = f"""

"""

In [ ]:
check("TODO 5a Gedanke vor jeder Aktion", lambda: "gedanke" in SYSTEM_REACT.lower())
check("TODO 5b Gliederung Befund, Ursache, Maßnahme", lambda: all(w in SYSTEM_REACT for w in ["Befund", "Ursache", "Maßnahme"]))
check("TODO 5c Aufhören und Offenes benennen", lambda: "offen" in SYSTEM_REACT.lower())

try:
    erg_react = agent(AUFGABE_M07, SYSTEM_REACT) if SYSTEM_REACT.strip() else None
except Exception:
    erg_react = None
if erg_react and trace_tabelle([]) is not None:
    display(trace_tabelle(erg_react["trace"]))
    print(f"\nStatus: {erg_react['status']} | Schritte: {erg_react['schritte']} | Referenzkosten: {kosten_cent(erg_react['tokens']):.2f} Cent")
    print("\n💬", erg_react["antwort"])

---
# Teil D · Plan-then-Execute

Bei ReAct entscheidet der Agent Schritt für Schritt. Bei **Plan-then-Execute** erstellt er **zuerst einen Plan** und arbeitet ihn dann ab. Das hilft bei Aufgaben mit vielen ähnlichen Teilschritten (etwa „alle acht Maschinen prüfen“) und macht das Vorgehen vorab prüfbar.

### TODO 6 · Planen
`plane(aufgabe)` ruft `client.chat.completions.parse(...)` mit `MODEL`, `max_tokens=500`, `response_format=Plan` und einer System-Nachricht auf, die die verfügbaren Tools nennt. Gebt `antwort.choices[0].message.parsed` zurück. Tipp: Die Tool-Namen und Beschreibungen stehen in `TOOLS`.

In [ ]:
class Plan(BaseModel):
    schritte: list[str] = Field(description="2 bis 6 kurze Schritte, jeweils mit dem Tool, das dafür nötig ist")

def plane(aufgabe):
    # TODO 6
    pass

In [ ]:
plan = plane(AUFGABE_M07)
check("TODO 6 Plan mit Schritten", lambda: isinstance(plan, Plan) and 1 <= len(plan.schritte) <= 8)
if plan:
    for i, s in enumerate(plan.schritte, 1):
        print(f"{i}. {s}")

### TODO 7 · Agent mit Plan
`agent_mit_plan(aufgabe, system)` erstellt einen Plan und startet den Agenten mit einer ersten Nachricht, die Aufgabe und nummerierten Plan enthält, plus dem Hinweis: „Arbeite den Plan ab. Weiche ab, wenn Ergebnisse es nahelegen.“ Gebt das Ergebnis von `agent` zurück und ergänzt `ergebnis["plan"]`.

In [ ]:
def agent_mit_plan(aufgabe, system, max_schritte=10):
    # TODO 7
    pass

In [ ]:
try:
    erg_plan = agent_mit_plan("Prüfe die Sensordaten aller acht Maschinen und nenne alle Auffälligkeiten mit Messwert und Grenzwert.", SYSTEM_REACT) if SYSTEM_REACT.strip() else None
except Exception:
    erg_plan = None
check("TODO 7 Agent mit Plan", lambda: erg_plan["status"] in ("fertig", "limit", "schleife") and len(erg_plan["plan"]) >= 1)
if erg_plan:
    print("Plan:", *erg_plan["plan"], sep="\n  ")
    print(f"\nStatus: {erg_plan['status']} | Schritte: {erg_plan['schritte']} | Referenzkosten: {kosten_cent(erg_plan['tokens']):.2f} Cent")
    print("💬", erg_plan["antwort"])

---
# Teil E · Messen: Hilft die Planung?

Sechs Agenten-Aufgaben mit Referenz, Pflichtbegriffen und erwarteten Tools. Bewertet wird wie in Woche 9 (Code-Checks plus Judge). Zusätzlich zählen wir **Schritte** und **Kosten** pro Aufgabe.

In [ ]:
AUFGABEN = [
    {'id': 'A1', 'aufgabe': 'Maschine M-07 meldet Überhitzung. Prüfe Status, Historie und Handbuch und schlage eine Maßnahme vor.', 'referenz': 'Am Prüfstand M-07 ist der Schaltschranklüfter ausgefallen (0 U/min), der Schaltschrank hat 58 °C bei Grenzwert 45 °C. Die elektrische Prüfung nach DGUV Vorschrift 3 ist seit Oktober 2026 fällig (letzte am 02.10.2025). Maßnahme: Prüfstand nicht weiter betreiben bzw. freischalten, Elektrofachkraft und Instandhaltung informieren, Lüfter tauschen, elektrische Prüfung nachholen, Ticket mit Priorität hoch.', 'muss': [['Lüfter'], ['Elektrofachkraft', 'elektrische Prüfung', 'DGUV']], 'tools': ['get_sensor_data', 'get_maintenance_history']},
    {'id': 'A2', 'aufgabe': 'Die Drehmaschine vibriert stärker als sonst. Finde die wahrscheinliche Ursache und sag, was zu tun ist.', 'referenz': 'M-03: Vibration 4,8 mm/s über dem Grenzwert 4,5. Wahrscheinliche Ursache sind die Spindellager: Rundlauf zuletzt 7 µm nahe am Grenzwert 8 µm, offenes Ticket S-021 mit geplantem Lagerwechsel. Maßnahme laut Handbuch: Drehzahl reduzieren, Instandhaltung informieren, Lagerwechsel (Spindellagersatz HTD-1100) vorziehen.', 'muss': [['Lager'], ['Drehzahl', 'Lagerwechsel', 'HTD-1100']], 'tools': ['get_sensor_data']},
    {'id': 'A3', 'aufgabe': 'Die Schweißzelle zeigt Fehler E21. Was bedeutet das, und gibt es Hinweise in den aktuellen Daten?', 'referenz': 'E21 bedeutet Übertemperatur im Kühlkreislauf des Brenners. Die Kühlmitteltemperatur liegt bei 68 °C über dem Grenzwert 60 °C. Maßnahme: Kühlflüssigkeit prüfen und auffüllen, Kühler auf Verschmutzung prüfen, Anlage abkühlen lassen.', 'muss': [['Kühl'], ['68']], 'tools': ['search_knowledge_base', 'get_sensor_data']},
    {'id': 'A4', 'aufgabe': 'Welche Maschinen zeigen gerade Auffälligkeiten in den Sensordaten? Nenne jeweils Messwert und Grenzwert.', 'referenz': 'Vier Maschinen: M-03 Vibration 4,8 mm/s (Grenzwert 4,5), M-05 Kühlmitteltemperatur Brenner 68 °C (60), M-07 Schaltschrank 58 °C (45) und Lüfter 0 U/min (min. 1500), M-08 Waschlauge 52 °C (min. 58). Alle anderen sind unauffällig.', 'muss': [['M-03'], ['M-05'], ['M-07'], ['M-08']], 'tools': ['get_sensor_data']},
    {'id': 'A5', 'aufgabe': 'Die Teile aus der Waschanlage bleiben fettig. Finde die Ursache und schlage eine Maßnahme vor.', 'referenz': 'Die Waschlauge hat nur 52 °C statt 60 °C (Mindestwert 58). Das Bad wurde am 20.09.2026 gewechselt, ist also nicht zu alt. Laut Handbuch Heizstab und Sicherung prüfen lassen.', 'muss': [['52'], ['Heizstab', 'Sicherung']], 'tools': ['get_sensor_data']},
    {'id': 'A6', 'aufgabe': 'Die Montagelinie B steht still. Was ist los?', 'referenz': 'Eine Montagelinie B gibt es nicht, nur die Montagelinie A (M-06), die unauffällig läuft. Der Assistent soll nachfragen, statt etwas zu erfinden.', 'muss': [['nicht', 'keine', 'gibt es nur', 'nur die Montagelinie A']], 'tools': []},
]

class Urteil(BaseModel):
    begruendung: str = Field(description="Ein bis zwei Sätze.")
    urteil: Literal["richtig", "teilweise", "falsch"]

JUDGE_SYSTEM = """Du bewertest Ergebnisse eines Instandhaltungs-Agenten anhand einer Referenz.
- richtig: Befund und Maßnahme stimmen in den wesentlichen Punkten mit der Referenz überein, nichts Wesentliches ist falsch oder erfunden.
- teilweise: Kern richtig, aber ein wesentlicher Punkt fehlt oder eine Aussage ist falsch.
- falsch: Kern falsch, erfunden oder Aufgabe nicht erledigt.
Länge ist kein Qualitätsmerkmal. Bewerte nur den Inhalt."""

import openai
def parse_mit_fallback(model, messages, schema, max_tokens=300):
    # Strukturierte Antwort als Pydantic-Objekt. Erst streng per json_schema. Lehnt das Modell das ab
    # (etwa DeepSeek V4 Flash), dann json_object mit dem Schema im Prompt. Pydantic validiert in beiden Fällen.
    try:
        return client.chat.completions.parse(model=model, max_tokens=max_tokens, messages=messages,
                                             response_format=schema).choices[0].message.parsed
    except openai.BadRequestError:
        hinweis = "Antworte ausschließlich mit einem JSON-Objekt nach diesem Schema, ohne weiteren Text: " + json.dumps(schema.model_json_schema(), ensure_ascii=False)
        msgs = [dict(m) for m in messages]
        if msgs and msgs[0]["role"] == "system":
            msgs[0]["content"] = msgs[0]["content"] + " " + hinweis
        else:
            msgs.insert(0, {"role": "system", "content": hinweis})
        a = client.chat.completions.create(model=model, max_tokens=max_tokens, messages=msgs, response_format={"type": "json_object"})
        text = (a.choices[0].message.content or "").strip().removeprefix("```json").removesuffix("```").strip()
        return schema.model_validate_json(text)

def judge(aufgabe, referenz, antwort):
    inhalt = f"<aufgabe>{aufgabe}</aufgabe>\n<referenz>{referenz}</referenz>\n<antwort>{antwort}</antwort>"
    return parse_mit_fallback(model=JUDGE_MODEL, max_tokens=300, schema=Urteil,
                              messages=[{"role": "system", "content": JUDGE_SYSTEM},
                                        {"role": "user", "content": inhalt}])

def wilson(k, n, z=1.96):
    p = k / n; nenner = 1 + z**2 / n
    mitte = (p + z**2 / (2 * n)) / nenner
    halb = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / nenner
    return max(0.0, mitte - halb), min(1.0, mitte + halb)

def bewerte_lauf(a, erg):
    antwort = erg["antwort"] or ""
    benutzt = [t["inhalt"].split("(")[0] for t in erg["trace"] if t["art"] == "Aktion"]
    inhalt_ok = all(any(b.lower() in antwort.lower() for b in g) for g in a["muss"])
    tools_ok = all(t in benutzt for t in a["tools"])
    u = judge(a["aufgabe"], a["referenz"], antwort) if antwort else Urteil(begruendung="keine Antwort", urteil="falsch")
    return {"id": a["id"], "status": erg["status"], "schritte": erg["schritte"], "kosten_cent": kosten_cent(erg["tokens"]),
            "tool_aufrufe": len(benutzt), "checks_ok": inhalt_ok and tools_ok, "urteil": u.urteil,
            "erfolg": erg["status"] == "fertig" and inhalt_ok and tools_ok and u.urteil == "richtig"}
print(len(AUFGABEN), "Aufgaben")

### TODO 8 · Zusammenfassen
`zusammenfassung(zeilen)` bekommt eine Liste von Ergebnissen aus `bewerte_lauf` und gibt ein Dictionary zurück mit:
- `erfolg`: Anzahl erfolgreicher Aufgaben
- `n`: Anzahl Aufgaben
- `intervall`: Wilson-Intervall
- `schritte_mittel`: mittlere Schritte
- `tool_aufrufe_mittel`: mittlere Tool-Aufrufe
- `kosten_mittel_cent`: mittlere Kosten
- `abbrueche`: Anzahl mit Status ungleich `fertig`

In [ ]:
def zusammenfassung(zeilen):
    # TODO 8
    pass

In [ ]:
_z = [{"erfolg": True, "schritte": 3, "tool_aufrufe": 4, "kosten_cent": 0.5, "status": "fertig"},
      {"erfolg": False, "schritte": 10, "tool_aufrufe": 12, "kosten_cent": 1.5, "status": "limit"}]
check("TODO 8a Zählungen", lambda: (lambda s: s["erfolg"] == 1 and s["n"] == 2 and s["abbrueche"] == 1)(zusammenfassung(_z)))
check("TODO 8b Mittelwerte", lambda: (lambda s: s["schritte_mittel"] == 6.5 and s["tool_aufrufe_mittel"] == 8 and math.isclose(s["kosten_mittel_cent"], 1.0))(zusammenfassung(_z)))

In [ ]:
if zusammenfassung(_z) is not None and SYSTEM_REACT.strip() and erg_plan is not None:
    tabellen = {}
    for name, laeufer in [("ReAct", lambda a: agent(a, SYSTEM_REACT)), ("ReAct + Plan", lambda a: agent_mit_plan(a, SYSTEM_REACT))]:
        zeilen = []
        for a in AUFGABEN:
            zeilen.append(bewerte_lauf(a, laeufer(a["aufgabe"])))
        s = zusammenfassung(zeilen)
        tabellen[name] = pd.DataFrame(zeilen).set_index("id")
        lo, hi = s["intervall"]
        print(f"{name:13s} Erfolg {s['erfolg']}/{s['n']} ({lo:.0%} bis {hi:.0%}) | Ø Schritte {s['schritte_mittel']:.1f} | "
              f"Ø Tool-Aufrufe {s['tool_aufrufe_mittel']:.1f} | Ø Referenzkosten {s['kosten_mittel_cent']:.2f} Cent | Abbrüche {s['abbrueche']}")
    display(pd.concat(tabellen, axis=1))

> **Zum Nachdenken:**
> - Hilft der Plan bei allen Aufgaben, oder nur bei bestimmten (etwa A4 mit acht Maschinen)?
> - Was kostet er zusätzlich?
> - Mit sechs Aufgaben sind die Intervalle sehr breit (Woche 9). Welche Aussage traut ihr euch trotzdem zu?

---
## Bonus

**B1 · Erfundene Tools.** Entfernt `get_sensor_data` aus `TOOLS` (nicht aus `FUNKTIONEN`) und stellt A2 erneut. Was tut der Agent ohne das passende Werkzeug?

**B2 · Evaluator-Optimizer.** Lasst einen zweiten Aufruf die Antwort des Agenten prüfen („Ist jede Aussage durch eine Beobachtung im Trace gedeckt?“) und bei Problemen eine überarbeitete Fassung erstellen.

**B3 · Kostenbremse.** Ergänzt im Loop ein Token-Budget: Überschreiten die Input-Tokens 30.000, bricht der Agent mit Status `budget` ab.

In [ ]:
# B1 / B2 / B3

## Projekt diese Woche

Im Notebook **`W10_Projekt_Agent.ipynb`** steckt euer Assistent jetzt in einem Agent-Loop mit Trace, Iterationslimit, Schleifenerkennung und optionalem Planungsschritt. Ergänzt **drei mehrschrittige Aufgaben** aus eurem Track in euer Testset, lasst das Eval laufen und weist **Erfolgsrate, Schritte und Kosten pro Aufgabe** aus.

## Zusammenfassung

- **Workflow:** Code bestimmt die Schritte. **Agent:** Das LLM bestimmt sie. Erst einen Workflow versuchen, einen Agenten nur, wenn er nötig ist.
- Der **Agent-Loop** ist die Tool-Schleife mit Trace, Iterationslimit und Schleifenerkennung.
- **ReAct:** Gedanke, Aktion, Beobachtung. Macht das Vorgehen zielgerichteter und den Trace lesbar.
- **Plan-then-Execute:** erst planen, dann ausführen. Hilft bei vielen gleichartigen Teilschritten und kostet einen zusätzlichen Aufruf.
- Ob etwas hilft, entscheidet das **Eval**: Erfolgsrate mit Intervall, Schritte, Kosten.

### Ausblick auf Woche 11
Lange Aufgaben und Gespräche werden teuer, und der Agent verliert Wichtiges aus dem Blick. Was gehört überhaupt in den Kontext? **Context Engineering und Memory.**